# Data Exploration

## Imports

In [ ]:
from snowflake.snowpark.session import Session
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T
from snowflake.snowpark.window import Window

import sys
import json
import pandas as pd
import numpy as np

## Create Snowpark Session

In [ ]:
session = get_active_session()

# Add a query tag to the session.
session.query_tag = {"origin":"sf_sit-is", "name":"e2e_snowflake_ml_cpu", "version":{"major":1, "minor":0}, "attributes":{"is_quickstart":3, "source":"notebook"}}

In [ ]:

print(f"Current Database and schema: {session.get_fully_qualified_current_schema()}")
print(f"Current Warehouse: {session.get_current_warehouse()}")
print(f"Current Role: {session.get_current_role()}")

Current Database and schema: "HOL_DB"."SCHEMA100"
Current Warehouse: "QUERY_WH"
Current Role: "DBA100"


## Snowpark DataFrames vs. Pandas DataFrames

In [ ]:
# Creating a Snowpark DataFrame
#snowpark_df = session.sql('select * from APPLICATION_RECORD')
snowpark_df = session.table('UTILITY.PUBLIC.APPLICATION_RECORD')
print(type(snowpark_df))

<class 'snowflake.snowpark.table.Table'>


A Snowpark DataFrame can be converted to a Pandas DataFrame. This will pull the data from Snowflake into the Python enviroment memory.

In [ ]:
# Creating a Pandas DataFrame
pandas_df = snowpark_df.to_pandas()
print(type(pandas_df))

<class 'pandas.core.frame.DataFrame'>


In [ ]:
# Compare size
print('Size in MB of Pandas DataFrame in Memory:\n', np.round(sys.getsizeof(pandas_df) / (1024.0**2), 2))
print('Size in MB of Snowpark DataFrame in Memory:\n', np.round(sys.getsizeof(snowpark_df) / (1024.0**2), 2))

Size in MB of Pandas DataFrame in Memory:
 251.15
Size in MB of Snowpark DataFrame in Memory:
 0.0


The only thing stored in a Snowpark DataFrame is the SQL needed to return data

In [ ]:
snowpark_df.queries

{'queries': ['SELECT  *  FROM (APPLICATION_RECORD)'], 'post_actions': []}

Showing a Snowpark DataFrame

In [ ]:
snowpark_df.show(5) 
#<- also possible
#snowpark_df.limit(5).to_pandas() # <- collects first 5 rows and displays as pandas-dataframe

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"ID"     |"CODE_GENDER"  |"FLAG_OWN_CAR"  |"FLAG_OWN_REALTY"  |"CNT_CHILDREN"  |"AMT_INCOME_TOTAL"  |"NAME_INCOME_TYPE"    |"NAME_EDUCATION_TYPE"          |"NAME_FAMILY_STATUS"  |"NAME_HOUSING_TYPE"  |"DAYS_BIRTH"  |"DAYS_EMPLOYED"  |"FLAG_MOBIL"  |"FLAG_WORK_PHONE"  |"FLAG_PHONE"  |"FLAG_EMAIL"  |"OCCUPATION_TYPE"  |"CNT_FAM_MEMBERS"  |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Simple Transformations
Select specific columns

In [ ]:
snowpark_df = snowpark_df.select('CODE_GENDER','NAME_INCOME_TYPE','DAYS_BIRTH',)
#snowpark_df = snowpark_df[['CODE_GENDER','NAME_INCOME_TYPE','DAYS_BIRTH']] # -> pandas-like selection
snowpark_df.show()

-------------------------------------------------------
|"CODE_GENDER"  |"NAME_INCOME_TYPE"    |"DAYS_BIRTH"  |
-------------------------------------------------------
|M              |Working               |-12005        |
|M              |Working               |-12005        |
|M              |Working               |-21474        |
|F              |Commercial associate  |-19110        |
|F              |Commercial associate  |-19110        |
|F              |Commercial associate  |-19110        |
|F              |Commercial associate  |-19110        |
|F              |Pensioner             |-22464        |
|F              |Pensioner             |-22464        |
|F              |Pensioner             |-22464        |
-------------------------------------------------------



To add a new column to a Snowpark DataFrame the **with_column** method can be used.  
In below example we are adding a neew column, AGE, that calculates the number of years that DAYS_BIRTH is.

In [ ]:
# Create a new column
# Formula: Absolute Value of DAYS_BIRTH divided by 365 days rounded down
snowpark_df = snowpark_df.with_column('AGE', F.floor(F.abs(F.col('DAYS_BIRTH')) / 365))
pandas_df3=snowpark_df.toPandas()
pandas_df3.head()

,CODE_GENDER,NAME_INCOME_TYPE,DAYS_BIRTH,AGE
0,M,Working,-12005,32
1,M,Working,-12005,32
2,M,Working,-21474,58
3,F,Commercial associate,-19110,52
4,F,Commercial associate,-19110,52
...,...,...,...,...
438552,M,Pensioner,-22717,62
438553,F,Working,-15939,43
438554,F,Commercial associate,-8169,22
438555,F,Pensioner,-21673,59


If we do not want to use specific columns we can use **drop** to remove those from a Snowpark DataFrame.  
**Note:** This is not removing them from the underlying table.

In [ ]:
# Drop a column
snowpark_df = snowpark_df.drop('DAYS_BIRTH')
snowpark_df.show()

------------------------------------------------
|"CODE_GENDER"  |"NAME_INCOME_TYPE"    |"AGE"  |
------------------------------------------------
|M              |Working               |32     |
|M              |Working               |32     |
|M              |Working               |58     |
|F              |Commercial associate  |52     |
|F              |Commercial associate  |52     |
|F              |Commercial associate  |52     |
|F              |Commercial associate  |52     |
|F              |Pensioner             |61     |
|F              |Pensioner             |61     |
|F              |Pensioner             |61     |
------------------------------------------------



To filter/select specific rows we use **filter**

In [ ]:
# Filter data
snowpark_df = snowpark_df.filter(F.col('NAME_INCOME_TYPE').in_(['Pensioner','Student']))
snowpark_df.show()

----------------------------------------------
|"CODE_GENDER"  |"NAME_INCOME_TYPE"  |"AGE"  |
----------------------------------------------
|F              |Pensioner           |61     |
|F              |Pensioner           |61     |
|F              |Pensioner           |61     |
|F              |Pensioner           |55     |
|F              |Pensioner           |61     |
|F              |Pensioner           |61     |
|F              |Pensioner           |61     |
|F              |Pensioner           |61     |
|F              |Pensioner           |61     |
|F              |Pensioner           |61     |
----------------------------------------------



To aggregate data the **group_by** method are used in combination with the **agg** method.

In [ ]:
snowpark_df = snowpark_df.group_by(['CODE_GENDER','NAME_INCOME_TYPE']).agg([F.avg('AGE').as_('AVG_AGE')])
snowpark_df.show()

--------------------------------------------------
|"CODE_GENDER"  |"NAME_INCOME_TYPE"  |"AVG_AGE"  |
--------------------------------------------------
|F              |Pensioner           |59.188624  |
|M              |Pensioner           |57.685482  |
|F              |Student             |46.090909  |
|M              |Student             |27.166667  |
--------------------------------------------------



In [ ]:
import streamlit as st
st.bar_chart(data =snowpark_df.to_pandas(), x='CODE_GENDER', y='AVG_AGE', color="NAME_INCOME_TYPE")

To sort the data in the dataframe **sort** is used.

In [ ]:
# Sort data
snowpark_df = snowpark_df.sort(F.col('AVG_AGE').desc())
snowpark_df.show()

--------------------------------------------------
|"CODE_GENDER"  |"NAME_INCOME_TYPE"  |"AVG_AGE"  |
--------------------------------------------------
|F              |Pensioner           |59.188624  |
|M              |Pensioner           |57.685482  |
|F              |Student             |46.090909  |
|M              |Student             |27.166667  |
--------------------------------------------------



## Simple Data Analysis
In this section we will use API Snowpark to do some basic analysis of our data.  
Start by creating a new Snowpark DataFrame

In [ ]:
snowpark_df = session.table("UTILITY.PUBLIC.APPLICATION_RECORD")

The **count** method on a DataFrame will return the number of rows

In [ ]:
# Number of rows in dataset
snowpark_df.count()

438557

If we want to filter out duplicated rows, keeping only one, we can use the **drop_duplicates** method.

In [ ]:
# Lets drop duplicates based on ID
snowpark_df = snowpark_df.drop_duplicates('ID')
snowpark_df.count()

438510

Duplicated rows are only filtered and we can see the logic for it by examining the SQL for the DataFrame, using ['queries'][0] will return the first SQL statement for the DataFrame

In [ ]:
print(snowpark_df.queries['queries'][0])

SELECT "ID", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "CNT_CHILDREN", "AMT_INCOME_TOTAL", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "DAYS_BIRTH", "DAYS_EMPLOYED", "FLAG_MOBIL", "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL", "OCCUPATION_TYPE", "CNT_FAM_MEMBERS" FROM ( SELECT "ID", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "CNT_CHILDREN", "AMT_INCOME_TOTAL", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "DAYS_BIRTH", "DAYS_EMPLOYED", "FLAG_MOBIL", "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL", "OCCUPATION_TYPE", "CNT_FAM_MEMBERS", row_number() OVER (PARTITION BY "ID"  ORDER BY "ID" ASC NULLS FIRST ) AS "122ox5t2g0" FROM APPLICATION_RECORD) WHERE ("122ox5t2g0" = 1 :: INT)


Using the **describe** method will return some basic statistics for all numeric and string columns.

In [ ]:
# Calculating various statistics per column
snowpark_df.describe().to_pandas()

,SUMMARY,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,mean,6.022035e+06,None,None,None,0.427381,1.875254e+05,None,None,None,None,-15998.022996,60566.188769,1.0,0.206128,0.287770,0.108200,None,2.194463
1,count,4.385100e+05,438510,438510,438510,438510.000000,4.385100e+05,438510,438510,438510,438510,438510.000000,438510.000000,438510.0,438510.000000,438510.000000,438510.000000,304317,438510.000000
2,min,5.008804e+06,F,N,N,0.000000,2.610000e+04,Commercial associate,Academic degree,Civil marriage,Co-op apartment,-25201.000000,-17531.000000,1.0,0.000000,0.000000,0.000000,Accountants,1.000000
3,stddev,5.714962e+05,None,None,None,0.724874,1.100893e+05,None,None,None,None,4185.016222,138770.072835,0.0,0.404523,0.452724,0.310633,None,0.897192
4,max,7.999952e+06,M,Y,Y,19.000000,6.750000e+06,Working,Secondary / secondary special,Widow,With parents,-7489.000000,365243.000000,1.0,1.000000,1.000000,1.000000,Waiters/barmen staff,20.000000


Using **group_by** and **agg** alows us to calculate the mean value of AMT_INCOME_TOTAL by NAME_INCOME_TYPE and CODE_GENDER. Using **sort** to return the data ordered by NAME_INCOME_TYPE in ascending order and AVG_INCOME by descending order.

In [ ]:
# Average Income per Income Type and Gender
analysis_df = snowpark_df.group_by(['NAME_INCOME_TYPE','CODE_GENDER']).agg([F.mean('AMT_INCOME_TOTAL').as_('AVG_INCOME')])
analysis_df = analysis_df.sort('NAME_INCOME_TYPE', F.col('AVG_INCOME').desc())
analysis_df.show()

-------------------------------------------------------------
|"NAME_INCOME_TYPE"    |"CODE_GENDER"  |"AVG_INCOME"        |
-------------------------------------------------------------
|Commercial associate  |M              |249208.08642289176  |
|Commercial associate  |F              |206579.17463258584  |
|Pensioner             |M              |169049.77416737832  |
|Pensioner             |F              |150729.61255448588  |
|State servant         |M              |237034.15414285715  |
|State servant         |F              |186152.9842904419   |
|Student               |F              |165272.72727272726  |
|Student               |M              |149250.0            |
|Working               |M              |202170.82427397132  |
|Working               |F              |168679.56899413437  |
-------------------------------------------------------------



**fillna** can be used to impute missing values with a new value. Below we are replacing missing values in the OCCUPATION_TYPE column with the mode value (snowpark_df[[F.mode('OCCUPATION_TYPE')]].collect()[0][0] calucaltes the mode and returns it as a value that is used)

In [ ]:
# Simple Missing Value Imputation
snowpark_df = snowpark_df.fillna(snowpark_df[[F.mode('OCCUPATION_TYPE')]].collect()[0][0], subset='OCCUPATION_TYPE')
snowpark_df.describe().to_pandas()

,SUMMARY,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,stddev,5.714962e+05,None,None,None,0.724874,1.100893e+05,None,None,None,None,4185.016222,138770.072835,0.0,0.404523,0.452724,0.310633,None,0.897192
1,count,4.385100e+05,438510,438510,438510,438510.000000,4.385100e+05,438510,438510,438510,438510,438510.000000,438510.000000,438510.0,438510.000000,438510.000000,438510.000000,438510,438510.000000
2,mean,6.022035e+06,None,None,None,0.427381,1.875254e+05,None,None,None,None,-15998.022996,60566.188769,1.0,0.206128,0.287770,0.108200,None,2.194463
3,min,5.008804e+06,F,N,N,0.000000,2.610000e+04,Commercial associate,Academic degree,Civil marriage,Co-op apartment,-25201.000000,-17531.000000,1.0,0.000000,0.000000,0.000000,Accountants,1.000000
4,max,7.999952e+06,M,Y,Y,19.000000,6.750000e+06,Working,Secondary / secondary special,Widow,With parents,-7489.000000,365243.000000,1.0,1.000000,1.000000,1.000000,Waiters/barmen staff,20.000000


The missing value handling logic is converted in SQL to *iff("OCCUPATION_TYPE" IS NULL, 'Laborers', "OCCUPATION_TYPE") AS "OCCUPATION_TYPE"*

In [ ]:
print(snowpark_df.queries['queries'][0])

SELECT "ID", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "CNT_CHILDREN", "AMT_INCOME_TOTAL", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "DAYS_BIRTH", "DAYS_EMPLOYED", "FLAG_MOBIL", "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL", iff("OCCUPATION_TYPE" IS NULL, 'Laborers', "OCCUPATION_TYPE") AS "OCCUPATION_TYPE", "CNT_FAM_MEMBERS" FROM ( SELECT "ID", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "CNT_CHILDREN", "AMT_INCOME_TOTAL", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "DAYS_BIRTH", "DAYS_EMPLOYED", "FLAG_MOBIL", "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL", "OCCUPATION_TYPE", "CNT_FAM_MEMBERS", row_number() OVER (PARTITION BY "ID"  ORDER BY "ID" ASC NULLS FIRST ) AS "122ox5t2g0" FROM APPLICATION_RECORD) WHERE ("122ox5t2g0" = 1 :: INT)


A DataFrame always has a schema with all columns and the data types for them

In [ ]:
snowpark_df.schema

StructType([StructField('ID', LongType(), nullable=True), StructField('CODE_GENDER', StringType(16777216), nullable=True), StructField('FLAG_OWN_CAR', StringType(16777216), nullable=True), StructField('FLAG_OWN_REALTY', StringType(16777216), nullable=True), StructField('CNT_CHILDREN', LongType(), nullable=True), StructField('AMT_INCOME_TOTAL', DoubleType(), nullable=True), StructField('NAME_INCOME_TYPE', StringType(16777216), nullable=True), StructField('NAME_EDUCATION_TYPE', StringType(16777216), nullable=True), StructField('NAME_FAMILY_STATUS', StringType(16777216), nullable=True), StructField('NAME_HOUSING_TYPE', StringType(16777216), nullable=True), StructField('DAYS_BIRTH', LongType(), nullable=True), StructField('DAYS_EMPLOYED', LongType(), nullable=True), StructField('FLAG_MOBIL', LongType(), nullable=True), StructField('FLAG_WORK_PHONE', LongType(), nullable=True), StructField('FLAG_PHONE', LongType(), nullable=True), StructField('FLAG_EMAIL', LongType(), nullable=True), Struct

We can itirate through the schema to get the columns of specific data types

In [ ]:
# Get all categorical columns
categorical_types = [T.StringType]
categorical_columns = [c.name for c in snowpark_df.schema.fields if type(c.datatype) in categorical_types]
categorical_columns

['CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE']

We can then use those to loop through and get the number of unique values

In [ ]:
# Number of unique values per categorical column
unique_values = []
for column in categorical_columns:
    unique_values.append([column, snowpark_df.select(column).distinct().count()])
pd.DataFrame(unique_values, columns=['COLUMN_NAME','NUM_UNIQUE_VALUES'])

,COLUMN_NAME,NUM_UNIQUE_VALUES
0,CODE_GENDER,2
1,FLAG_OWN_CAR,2
2,FLAG_OWN_REALTY,2
3,NAME_INCOME_TYPE,5
4,NAME_EDUCATION_TYPE,5
5,NAME_FAMILY_STATUS,5
6,NAME_HOUSING_TYPE,6
7,OCCUPATION_TYPE,18


## Persist Transformations

If we want to save the changes we can either save it as a table, meaning the SQL generated by the DataFrame is executed and the result is stored in a table or as a view where the DataFrame SQL will be the definition of the view.  
**save_as_table** saves the result in a table, if **mode='overwrite'** then it will also replace the data that is in it.

In [ ]:
snowpark_df.write.save_as_table(table_name='MY_FIRST_ANALYSIS', mode='overwrite')
session.table('MY_FIRST_ANALYSIS').limit(5).to_pandas()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5633001,M,Y,N,0,225000.0,Working,Incomplete higher,Married,House / apartment,-14280,-258,1,1,0,0,Laborers,2.0
1,6460419,F,N,Y,1,67500.0,Commercial associate,Secondary / secondary special,Separated,House / apartment,-13510,-3852,1,0,0,0,Sales staff,2.0
2,6448814,M,N,Y,0,166500.0,Commercial associate,Secondary / secondary special,Civil marriage,House / apartment,-8137,-610,1,1,1,0,Sales staff,2.0
3,5090669,F,Y,Y,0,225000.0,Working,Secondary / secondary special,Married,House / apartment,-17821,-1830,1,0,0,0,Laborers,2.0
4,5069096,F,N,Y,2,270000.0,Commercial associate,Higher education,Married,House / apartment,-11506,-2331,1,0,0,0,Laborers,4.0


In [ ]:
session.close()